# Percentils, Bootstrapping and Confidence Intervals

This notebook reviews Lecture 23: Confidence Intervals and Lecture 24: Interpreting Confidence. 

**Helpful Resource:**
- [Python Reference](http://data8.org/sp22/python-reference.html): Cheat sheet of helpful array & table methods used in Data 8!

**Recommended Readings**: 
- [Percentiles](https://inferentialthinking.com/chapters/13/1/Percentiles.html)
- [Bootstrap](https://inferentialthinking.com/chapters/13/2/Bootstrap.html)
- [Confidence Intervals](https://inferentialthinking.com/chapters/13/3/Confidence_Intervals.html)

In [ ]:
from datascience import *
%matplotlib inline
path_data = '../../../assets/data/'
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import numpy as np
from math import ceil

## Estimation

Recap, a parameter is a numerical value associated with population, such as mean, median, standard deviation, etc. In other words, a parameter is a charactistic of a population.

Almost always, parameters that we are interested are unknown/hidden in the real world population. But we can use percentiles, bootstrapping and confidence interval to estimate the parameters of a population. Percentiles are mathematical tools. Bootstrapping is a simulation tool. Confidence interval is a estimation tool.

"The value of any statistic depends on the sample, and the sample is based on random draws. So every time data scientists come up with an estimate based on a random sample, they are faced with a question:"

**“How different could this estimate have been, if the sample had come out differently?”**

### Percentiles

Percentiles are positions from 0% to 100%.

Let p be a number between 0 and 100. The pth percentile of a collection is the smallest value in the collection that is at least as large as p% of all the values.

By this definition, any percentile between 0 and 100 can be computed for any collection of values, and it is always an element of the collection.

In practical terms, suppose there are n elements in the collection. To find the pth percentile:

- Sort the collection in increasing order.
- Find p% of n: (p/100) * n. Call that k.
- If k is an integer, take the kth element of the sorted collection.
- If k is not an integer, round it up to the next integer, and take that element of the sorted collection.

### The Bootstrap: Resampling from the Sample

#### How bootstapping works?
By the law of averages, the distribution of the original sample is likely to resemble the population, and the distributions of all the “resamples” are likely to resemble the original sample. So the distributions of all the resamples are likely to resemble the population as well.

1. Treat the original sample as if it were the population.
2. Draw from the sample, at random with replacement, the same number of times as the original sample size.

**Use `tbl.sample()` with replacement to redraw samples from the sample distribution**

We can use bootstrapping to determine confidence interval.

### Percentiles Example

Use the student data to identify the 20th, 40th, 60th, and 80th quartile of the height. Calculate your answers in two ways: once using the `percentile` function provided by the `datascience` package, and a second time by using the definition of percentiles. The position $L$ in a sorted list for the k-th percentile is given by: 

Find p% of n: 

$k = \frac{p}{100} * n$

If $k$ is not an integer, you can use the `ceil` function in the `math` package to round $k$ up to the next greater integer. 

In [ ]:
students = Table.read_table('student_data.csv')
students.show(3)

In [ ]:
students.column("AGE")

#### `percentile()` datascience function

`percentile(percentile, array)`

Returns the corresponding percentile of an array.

In [ ]:
# sort the array and return the corresponding percentile of an array.
percentile(10, students.column("AGE"))

In [ ]:
# 20th, 40th, 60th, and 80th

print("20th :", percentile(20, students.column("AGE")))
print("40th :", percentile(40, students.column("AGE")))
print("60th :", percentile(60, students.column("AGE")))
print("80th :", percentile(80, students.column("AGE")))

In [ ]:
# compute percentiles using formula
ages = students.sort("AGE").column("AGE")
ages

In [ ]:
_20th = 20 / 100 * len(ages)
print("20th :", ages.item(ceil(_20th)))

_40th = 40 / 100 * len(ages)
print("40th :", ages.item(ceil(_40th)))

_60th = 60 / 100 * len(ages)
print("60th :", ages.item(ceil(_60th)))

_80th = 80 / 100 * len(ages)
print("80th :", ages.item(ceil(_80th)))

The `boxplot` shows some of the important percentiles from a given set of data. Run the following code to examine the boxplot of the student height data. How do your percentiles compare with these? 

In [ ]:
# boxplot() captures the max and min boundary, outliners, 
#   25th, 50th and 75th percentiles and the middle 50% of the data
age_array = students.column('AGE')
plots.boxplot(age_array, tick_labels=['Student Age'])

In [ ]:
percentile(50, students.column("AGE"))

In [ ]:
height_array = students.column('HEIGHT')
plots.boxplot(height_array, tick_labels=['Student Height'])

### Bootstrapping Example Tech Salaries

Parameter: the median of the tech salaries

In [ ]:
tech_salaries = Table.read_table("tech_salaries.csv")
tech_salaries.show(3)

In [ ]:
# check max and min salaries
(tech_salaries.column('Salary').max(), tech_salaries.column('Salary').min())

In [ ]:
tech_salaries.sort('Salary', descending=False)

In [ ]:
# check the overall data distribution
tech_salaries.hist('Salary', bins=np.arange(30000, 700001, 25000))

In [ ]:
data_from_histogram = tech_salaries.where(
        'Salary', are.above_or_equal_to(30000)
    ).where(
        'Salary', are.below_or_equal_to(700000)
    )#.num_rows

data_from_histogram

As mentioned previously, bootstrapping, percentiles and confidence interval are tools to estimate the parameters of a population. Almost always, we don't the real parameters. But, since we have the entire dataset, we can take a look.

In [ ]:
# check the real salary median from the dataset
salary_median = percentile(50, tech_salaries.column('Salary'))
salary_median

In [ ]:
# let's look at the salary median within the range in the histogram 
percentile(50, data_from_histogram.column('Salary'))

#### A Random Sample and an Estimate

Draw a large sample of high tech salaries at random **without replacement**. We draw without replacement because this is our original sample, we will be using it for bootstrapping. We want each data point is unique from the population.

In [ ]:
original_sample = tech_salaries.sample(700, with_replacement=False)
original_sample.show(3)

In [ ]:
# check the max and min the original_sample
(original_sample.column('Salary').max(), original_sample.column('Salary').min())

In [ ]:
original_sample.select('Salary').hist(bins=np.arange(30000, 700001, 25000))

In [ ]:
# check the salary median from the first sample
original_sample_median = percentile(50, original_sample.column('Salary'))
original_sample_median

The sample size is large. By the law of averages, the distribution of the sample resembles that of the population. 

### Bootstrap: Resampling from the Sample

We are going to resample from the original sample that we drew from the entire population. It is call resampling from the sample. The subsequence resamplings in a simulation will also be drawn from the original sample, not the population.

1. Treat the original sample as if it were the population.
2. Draw from the sample, at random with replacement, the same number of times as the original sample size (resampling the entire original sample in the bootstrapping simulation).

In [ ]:
# let see how it goes with the first resampling
#   by default, tbl.sample() draws with replacement and the sample size is the size of the table
re_sample_1 = original_sample.sample()
re_sample_1.show(3)

In [ ]:
# let's check the max and min
(re_sample_1.column('Salary').max(), re_sample_1.column('Salary').min())

In [ ]:
# check the histogram
re_sample_1.select('Salary').hist(bins=np.arange(30000, 700001, 25000))

In [ ]:
re_sampled_median_1 = percentile(50, re_sample_1.column('Salary'))
re_sampled_median_1

Becasue of the law of averages, the distribution of the original sample is likely to resemble the population, and the distributions of all the “resamples” are likely to resemble the original sample.

In [ ]:
print(f"salary_median:", salary_median, "original_sample_median:", original_sample_median, "re_sampled_median_1:", re_sampled_median_1)

We can keep resampling from the original sample over and over again.
Let's create functions to simulate bootstripping.

In [ ]:
# function to bootstrap 1 median
def one_bootstrap_median():
    re_sampled_table = original_sample.sample()
    ###*** for demo only, remove the print statement to continue simulation
    #print(f"max: {re_sampled_table.column('Salary').max()}, min: {re_sampled_table.column('Salary').min()}")
    bootstrapped_median = percentile(50, re_sampled_table.column('Salary'))
    return bootstrapped_median

In [ ]:
one_bootstrap_median()

### Bootstrap Empirical Distribution of the Sample Median

In [ ]:
repeats = 10000
bootstrap_medians = make_array()
for i in np.arange(repeats):
    bootstrap_medians = np.append (bootstrap_medians, one_bootstrap_median())

In [ ]:
(max(bootstrap_medians), min(bootstrap_medians))

In [ ]:
re_sampled_medians = Table().with_column('Bootstrap Sample Median', bootstrap_medians)
median_bins=np.arange(168000, 202000, 2000)
re_sampled_medians.hist(bins = median_bins)

# Plotting parameters; you can ignore this code
plots.ylim(-0.000005, 0.00014)
#parameter_green = '#32CD32'
#plots.scatter(salary_median, 0, color=parameter_green, s=40, zorder=2);
#plots.axvline(salary_median, color='red', linestyle='--', linewidth=2, label='Observed Data')
plots.axvline(original_sample_median, color='red', linestyle='--', linewidth=2, label='Observed Data')
plots.legend()

In [ ]:
# let's check it with boxplot()
bstrapped_sample_median_array = re_sampled_medians.select('Bootstrap Sample Median')
plots.boxplot(bstrapped_sample_median_array, tick_labels=['Bootstrap Sample Median'])

### Do the Estimates Capture the Parameter? Confidence Interval

The observed statistic is the median calculated from the original sample which is the red dash vertical line on the histogram.

The estimates are the medians (bootstrapped statistics) calculated from the bootstrapping simulation.

Parameter is again the hidden the median (test statistic) in the real world population.

So based on the observed statistic and the bootstrapped statistics, we use a confidence interval to evaluate where the true population parameter is hidden.

We use a confidence interval to measure the random sampling variability in our data.

In other words, a confidence interval provides a range of estimated parameter values instead of one single statistic value, acting as a wider net to predict the unknown parameter in the real-world population

We typically try to capture the "middle 95%".

In [ ]:
left = percentile(2.5, bootstrap_medians)
left

In [ ]:
right = percentile(97.5, bootstrap_medians)
right

In [ ]:
re_sampled_medians.hist(bins = median_bins)
plots.ylim(-0.000005, 0.00014)
#plots.scatter(salary_median, 0, color=parameter_green, s=40, zorder=2);
#plots.axvline(salary_median, color='red', linestyle='--', linewidth=2, label='Observed Data')
#plots.axvline(original_sample_median, color='red', linestyle='--', linewidth=2, label='Observed Data')
plots.axvline(original_sample_median, color='red', linestyle='--', linewidth=2, label='Observed Data from Original Sample')
plots.plot([left, right], [0, 0], color='yellow', lw=3, zorder=1)
plots.legend()

The yellow floor bar is the confidence interval. It looks pretty good traversing the main body of the histogram.

### Estimating a Population Average

We use the student data to create a bootstrap confidence interval for the average age in the student population in SRJC. In other words, we try to estimate SRJC students average age by bootstrapping the sample data in the given student dataset.

In [ ]:
students = Table.read_table('student_data.csv')
students.show(3)

In [ ]:
(students.column('AGE').max(), students.column('AGE').min())

In [ ]:
#students.hist('AGE', bins=np.arange(12, 52, 1))
students.hist('AGE')

In [ ]:
# let's look at the average in our sample data
original_sample_mean = np.average(students.column('AGE'))
original_sample_mean

But we don't know the average age of the entire student population in SRJC and that is the unknown parameter we are trying to find out.

First, we creat a function to bootstrap one mean.

In [ ]:
def one_bootstrap_mean():
    re_sample = students.sample()
    return np.average(re_sample.column('AGE'))

Second, repeat the bootstrip many times.

In [ ]:
# Generate means from 5000 bootstrap samples
repeats = 5000
bootstrap_means = make_array()
for i in np.arange(repeats):
    bootstrap_means = np.append(bootstrap_means, one_bootstrap_mean())

Then, calculate the "middle 95%".

In [ ]:
# Get the endpoints of the 95% confidence interval
left = percentile(2.5, bootstrap_means)
right = percentile(97.5, bootstrap_means)

make_array(left, right)

In [ ]:
re_sampled_means = Table().with_columns(
    'Bootstrap Sample Mean', bootstrap_means
)
re_sampled_means.hist(bins=15)
plots.plot([percentile(5, bootstrap_means), percentile(95, bootstrap_means)], [0, 0], color='pink', lw=45, label='90%');
plots.plot([left, right], [0, 0], color='yellow', lw=30, label='95%');
plots.plot([percentile(0.5, bootstrap_means), percentile(99.5, bootstrap_means)], [0, 0], color='skyblue', lw=15, label='99%');
plots.legend()

We compare 90%, 95% and 99% confidence intervals.

### Estimating a Population Proportion
#### Proportion is also the Mean

A regional environmental agency wants to estimate the true proportion of public electric vehicle (EV) charging stations in a state that are broken or malfunctioning on any given day. They inspect a random sample of 250 stations across the state. In their dataset, they record a 1 if the station is broken and a 0 if it is working perfectly. Out of the 250 audited stations, 35 are broken.

In [ ]:
ev_audit = Table.read_table('ev_charging_audit.csv')
ev_audit.show(3)

In [ ]:
# the value is stored as float type, displayed as integer
ev_audit.column('Is_Broken').dtype

In [ ]:
broken = ev_audit.column('Is_Broken')
observed_broken_proportion = np.count_nonzero(broken) / len(broken)
observed_broken_proportion

In [ ]:
np.mean(ev_audit.column('Is_Broken'))

In [ ]:
# np.average() has weights option
np.average(ev_audit.column('Is_Broken'))

In [ ]:
# create on function to bootstraps the sample and 
#   returns the proportion of broken ev station in the bootstrapped sample.
def one_bootstrap_proportion():
    re_sample = ev_audit.sample()
    broken = re_sample.column('Is_Broken')
    return np.count_nonzero(broken) / len(broken)

In [ ]:
# check the result of one re-sampling
one_bootstrap_proportion()

In [ ]:
# Generate proportions from many bootstrap samples
bootstrap_proportions = make_array()
repeats = 5000
for i in np.arange(repeats):
    bootstrap_proportions = np.append(bootstrap_proportions, one_bootstrap_proportion())

In [ ]:
Table().with_columns('bootstrap_proportions', bootstrap_proportions).hist()

In [ ]:
# let's compute the endpoints of the 95% confidence interval ('middle 95%')

left = percentile(2.5, bootstrap_proportions)
right = percentile(97.5, bootstrap_proportions)

make_array(left, right)

In [ ]:
re_sampled_proportions = Table().with_columns(
    'Bootstrap Sample Proportion', bootstrap_proportions
)
re_sampled_proportions.hist(bins=15)
plots.plot([left, right], [0, 0], color='yellow', lw=10);
plots.axvline(observed_broken_proportion, color='red', linestyle='--', linewidth=2, label='Observed Data from Original Sample')
plots.legend()
print(f"Audit Sample Size (n): {ev_audit.num_rows} stations")
print(f"Number of Broken Chargers found: {np.count_nonzero(ev_audit.column('Is_Broken') == 1)}")
print(f"Observed Broken Proportion (Sample Mean): {observed_broken_proportion:.4f} (or {observed_broken_proportion*100:.1f}%)\n")

The observed broken proportion is inside the 95% confidence interval.

### What a Confidence Interval Can Do?

1. provide a range of estimated parameter values instead of one single statistic value, acting as a wider net to predict the unknown parameter in the real-world population
2. measure the precision and reliability of the data because an estimated range is much more reliable than a single data point
3. test hypotheses that are claimed by others

### Using a Confidence Interval to Test Hypotheses

Null hypothesis: The true population proportion of broken chargers is greater than or equal to 0.10 (10%)

Alternative hypothesis: The true population proportion of broken chargers is less than 0.10 (10%)

Let's check where the null model resides in the histogram.

In [ ]:
null_model = 0.10
re_sampled_proportions.hist(bins=15)
plots.plot([left, right], [0, 0], color='yellow', lw=10);
plots.axvline(null_model, color='red', linestyle='--', linewidth=2, label='Null Model')
plots.legend()

Because the 0.10 (10%) claim value is captured within the confidence interval, we fail to reject the null hypothesis.

### Confidence Interval for Global Warming Survey

A survey on 2,251 randomly selected individuals conducted in October 2020 found that 1,328 answered Yes to the question “Is there solid evidence of global warming?”

What percentage of Americans believe in global warming?  



In [ ]:
warming = Table.read_table('global_warming_survey_2020.csv')
warming.show(3)

In [ ]:
# let's create a function to calculate a mean what works for all data tables
def one_bootstrap_mean(table, coluum_name):
    re_sample = table.sample()
    return np.average(re_sample.column(coluum_name))

In [ ]:
# check the result of one re-sampling
one_bootstrap_mean(warming, 'Answered_Yes')

In [ ]:
# Generate means from 5000 bootstrap samples
repeats = 5000
bootstrap_means = make_array()
for i in np.arange(repeats):
    bootstrap_means = np.append(bootstrap_means, one_bootstrap_mean(warming, 'Answered_Yes'))

In [ ]:
# Get the endpoints of the 95% confidence interval
left = percentile(2.5, bootstrap_means)
right = percentile(97.5, bootstrap_means)

make_array(left, right)

In [ ]:
re_sampled_means = Table().with_columns(
    'Bootstrap Sample Mean', bootstrap_means
)
re_sampled_means.hist(bins=15)
plots.plot([percentile(5, bootstrap_means), percentile(95, bootstrap_means)], [0, 0], color='pink', lw=45, label='90%');
plots.plot([left, right], [0, 0], color='yellow', lw=30, label='95%');
plots.plot([percentile(0.5, bootstrap_means), percentile(99.5, bootstrap_means)], [0, 0], color='skyblue', lw=15, label='99%');
plots.legend()

### Test Hypotheses

Assume we have the following test hypotheses:

Null hypothesis: The true population proportion of individuals who believe global warming is happening is exactly 0.55 (55%). Any variation between the sample proportion and 0.55 is due to random sampling variability.

Alternative hypothesis: The true population proportion of individuals who believe global warming is happening is different from 0.55. The deviation in the survey dataset is a structural reality issue.

In [ ]:
null_model = .55
re_sampled_means.hist(bins=15)
plots.plot([percentile(5, bootstrap_means), percentile(95, bootstrap_means)], [0, 0], color='pink', lw=45, label='90%');
plots.plot([left, right], [0, 0], color='yellow', lw=30, label='95%');
plots.plot([percentile(0.5, bootstrap_means), percentile(99.5, bootstrap_means)], [0, 0], color='skyblue', lw=15, label='99%');
plots.axvline(null_model, color='red', linestyle='--', linewidth=2, label='Null Model')
plots.legend()

Because the 0.55 (55%) claim value is completely outside and far left of the confidence interval, we reject the null hypothesis in favor of the alternative.

### Confidence Interval for a Difference of Two Means

Using the NBA data to construct a 95% confidence interval for the average salary of players in 2020 and compare it with the 2010 season.

In [ ]:
nba = Table.read_table('nba_salaries.csv')
nba.show(5)

In [ ]:
nba_2020 = nba.where('season', are.equal_to(2020))
nba_2010 = nba.where('season', are.equal_to(2010))
nba_2020.num_rows, nba_2010.num_rows

In [ ]:
nba_2020.show(5)

In [ ]:
# we perviously created a function to do one bootstrap mean 
#(one_bootstrap_mean(nba_2020, 'salary'), one_bootstrap_mean(nba_2010, 'salary'))

original_sample_mean_2020 = one_bootstrap_mean(nba_2020, 'salary') / 1000000
original_sample_mean_2010 = one_bootstrap_mean(nba_2010, 'salary') / 1000000
original_sample_mean_diff = (original_sample_mean_2020 - original_sample_mean_2010)
original_sample_mean_diff

In [ ]:
# let's create function simulate the bootstrap the difference of 2 means many times
#   the function parameter list: table, column name and repeats
def simulate_bootstrap_mean(table, column_name, repeats):
    bootstrap_means = make_array()
    for i in np.arange(repeats):
        re_sampled_mean = one_bootstrap_mean(table, column_name)
        bootstrap_means = np.append(bootstrap_means, re_sampled_mean)
    return bootstrap_means

In [ ]:
# bootstrap the mean salary for 2020 and 2010
means_2020 = simulate_bootstrap_mean(nba_2020, 'salary', 5000)
means_2010 = simulate_bootstrap_mean(nba_2010, 'salary', 5000)

(means_2020, means_2010)

In [ ]:
# create a table of 2 means - convert the salary in 1 million unit
means_2020_2010 = Table().with_columns('mean 2020', means_2020 / 1000000, 'mean 2010', means_2010 / 1000000)
means_2020_2010.show(5)

In [ ]:
means_2020_2010.hist('mean 2020', 'mean 2010')

In [ ]:
means_2020_2010 = means_2020_2010.with_columns(
    'diff mean', means_2020_2010.column('mean 2020') - means_2020_2010.column('mean 2010'))
means_2020_2010.show(5)

In [ ]:
left = percentile(2.5, means_2020_2010.column('diff mean'))
right = percentile(97.5, means_2020_2010.column('diff mean'))

(left, right)

In [ ]:
print(f"Observed 2020 Mean Salary: ${original_sample_mean_2020:.3f}M")
print(f"Observed 2010 Mean Salary: ${original_sample_mean_2010:.3f}M")
print(f"Observed Difference Salary: ${original_sample_mean_diff:.3f}M\n")
means_2020_2010.hist('diff mean')
plots.axvline(original_sample_mean_diff, color='red', linestyle='--', linewidth=2, label='Observed Mean Diff')
plots.plot([left, right], [0, 0], color='yellow', lw=15, label='95%');
plots.legend()